In [9]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

# 기본적으로 현재 작업 디렉터리에 있는 .env 파일을 자동으로 찾아서 로드함.
load_dotenv()

True

In [10]:
# 검색 도구 정의.
# max_results 를 통해 검색 결과를 최대 2개까지 반환하도록 설정. 그리고 랭그래프에 넘기기 위해 도구들을 리스트로 묶어줌.
tool = TavilySearch(max_results = 2)
tools = [tool]

#tool.invoke("랭그래프에서 노드란 무엇인가?")

# 실행결과는 다음과 같이 다양한 정보를 포함하는 딕셔너리 형태로 반환됨.
# .query : 실제로 검색한 질문
# .follow_up_questions : 추가 질문이 필요한 경우 해당 질문이 여기에 나타남.
# .answer : 도구가 직접 생성한 최종 답변, 
# .images : 관련 이미지가 있으면 여기에 포함됨
# .results
#   .url : 참고할 만한 웹 문서 링크.
#   .title : 해당 문서의 제목.
#   .content : 문서의 주요 요약이나 본문 일부
#   .score: 검색 결과의 신뢰도/유사도 점수(0~1 사이, 높을 수록 관련성이 높음.)
#   .raw_content : 원본 전체 내용(여기서는 제공되지 않음)
#   .response_time : 검색에 걸린 시간(초) 

In [11]:
# 그래프에 도구 연동.
# 다음 단계에서는 StateGraph(상태 그래프)에 검색 도구를 연동하는 과정을 살펴봄.

# 우선 사용할 LLM 을 초기화 함. 그리고 bind_tools() 메서드로 해당 LLM이 어떤 도구들을 사용할 수 있는지 명시함.
# 이렇게 하면 LLM은 대화 도중 필요에 따라 검색 도구를 직접 호울할 수 있게 됨.
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1")
llm_with_tools = llm.bind_tools(tools)

In [12]:
# 이제 챗봇 노드를 정의할 때 기존 LLM 대신 llm_with_tools 를 사용함.
# 다음 예시는 챗봇이 상태(State)내 메시지 이력을 입력받아 필요 시 도구를 호출해 최신 정보까지 반영한 답변을 생성하는 구조를 보여줌.
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 정의된 챗본 노드를 그래프에 추가함.
graph_builder.add_node("chatbot", chatbot)

# 이 과정을 통해 랭그래프내 챗봇은 단순 대화뿐만 아니라 실시간 웹 검색 결과까지 반영한 고도화된 답변을 생성할 수 있음.

In [13]:
# 이제 도구 노드와 조건부 흐름을 추가해 챗봇이 도구 사용이 필요한지 여부에 따라 자동으로 검색 기능을 호출하고, 그 결과를 다시 대화에 반영할 수있도록 워크플로우를 확장해 보겠음.
# 도수 실행 노드 만들기.
# 챗봇이 도구를 호출할때 실제로 이 도구를 실행할 별도의 노드를 만들어야 함. 랭그래프는 이를 위해 두가지 방법을 제공함.

# 1. 직접 노드를 정의하는 방법.
import json
# ToolMessage 
# LLM이 호출한 도구(Tool)의 실행 결과를 다시 LLM에게 전달할 때 사용하는 메시지 객체임.
# 파이썬 함수나 API를 실행한 후, 그 결과 값을 다시 LLM을 보낼때 사용하는 규격임.
# 속성
#   . content : 도구가 실제 수행하고 반환한 결과 데이터(대개 문자열)
#   . tool_call_id: 어떤 도구 호출 요청에 대한 답변인지 매칭해 주는 고유 ID(LLM이 처음에 준 ID 그대로 사용)

###############################################################################################################
#from langchain_core.messages import ToolMessage

#class BasicToolNode:
#    """챗봇이 요청한 도구를 실행하는 도구입니다."""

#    def __init__(self, tools: list):
#        self.tools_by_name = {tool.name for tool in tools}

#    def __call__(self, inputs: dict, outputs: list):
#        messages = inputs.get("messages", [])
#        if not messages:
#            raise ValueError("입력된 상태에서 메시지를 찾을 수 없습니다.")
#            outputs = []

#        for tool_call in messages[-1].tool_calls:
#            tool_result = self.tools_by_name[tool_call["name"]].invoke(tool_call["args"])

#            outputs.append(
#                ToolMessage(
#                    content = json.dump(tool_result), #json.dump() : 파이썬 객체 (딕셔너리, 리스트 등)를 JSON 형식의 "문자열(String)"로 변환해 주는 함수.
#                    name = tool_call["name"],
#                    tool_call_id = tool_call["id"]
#                )
#            )

#        return {"messages": outputs}

#tool_node = BasicToolNode(tools=[tool])
##############################################################################################################

# 2. 랭그래프가 미리 만들어둔 ToolNode 를 활용하는 방법.
#   직접 노드를 정의하는 방법은 호출 메시지를 직접 관리할 수 있는 장점이 있지만 코드가 길어지고 복잡해질 수 있음.
#   그래서 일반적으로 랭그래프가 제공하는 미리 만들어진 prebuilt 라이브러리인 ToolNode 를 활용하는 것이 쉽고 효율적임.

from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools=[tool])

# 이 방법은 내부적으로 도구 호출 처리과정을 모두 포함하고 잇어 직접 구현할 필요없이 간단히 사용할수 있다는 장점이 있음.
# 앞의 두가지 방법으로는 도구 노드를 만들었다면 다음 코드를 통해 그래프에 도구 노드를 추가할 수 있음.
graph_builder.add_node("tools", tool_node)



In [14]:
# 조건부 엣지 정의하기.
# 이제 그래프에 추가한 도구 노드를 실제로 사용할 지를 결정하는 조건부 엣지를 정의함.
# 조건부 엣지는 특정 조건에 따라 노드 사이의 이동 경로를 결정함. 보통 if 조건문을 사용해 상태에 따라 다음에 실행할 노드를 지정함.

# 조건부 엣지는 다음 2가지 방식으로 정의할 수 있음.
# 1. 직접 조건부 엣지를 정의하는 방법.

# 아래의 코드는 route_tools  함수는 챗봇의 가장 최근 메시지를 확인해 도구 호출(tool_calls)여부에 따라 다음 이동 노드를 결정함. 이 과정을 정리하자면 다음과 같음.
###################################################################################################
#def route_tools(state: State):
#    """마지막 챗봇 메시지에서 도구 호출(tool_calls)이 있는지 확인하여
#    있으면 'tools' 노드로 없으면 END(종료)로 이동함."""

#    ai_message = state["messages"][-1]

#    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
#        return "tools" # 도구 호출이 있으면 도구 노드로 이동.

#    return END # 도구 호출이 없으면 종료

#graph_builder.add_conditional_edges(
#    "chatbot",
#    route_tools,
#    {"tools": "tools", END: END}
#)
###################################################################################################

# route_tools 함수는 챗봇 노드가 생성한 최신 메시지를 확인함.
# 메시지 내에 tool_calls 가 있는지 검사함.
#   . 만약 도구 호출(tool_calls)이 있다면 다음 노드로 "tools" 노드를 반환해 도구를 실행하도록 함.
#   . 도구 호출이 없다면 "END"를 반환해 처리를 종료하도록 함.

# 이처럼 챗봇이 도구 사용 여부를 판단해 그 결과에 따라 도구 실행 노드로 이동하거나 처리를 마무리하는 구조임.
# 이 방법은 조건 판단 로직을 직접 명시적으로 작성할 수 있어 유연성이 뛰어나지만 코드가 길어질수록 있음.

# 따라서 일반적으로 랭그래프가 제공하는 미리 만들어진(prebuilt) 조건이 엣지인 tools_condition 을 활용하면 더 간편하게 설정할 수 있음.

# 2. 랭그래프가 미리 만들어 둔 tools_condition 을 활용하는 방법.

# 간단하게 tools_condition 이라는 함수를 사용해 조건부 엣지를 정의할 수 있음.
# 이 함수는 챗봇의 마지막 메시지를 확인해 도구 사용 요청이 포함돼 있으면 노두 노드로 이동하고, 그렇지 않으면 챗봇이 직접 응답을 완료하도록 함.
from langgraph.prebuilt import tools_condition

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

# 조건부 엣지를 정의한  후에는 이 조건부 엣지를 그래프에 연결해 실제 노드 간의 이동이 조건에 따라 이뤄지도록 만들어 줌.
# 이를 통해 챗봇 노드가 도구를 호출해야 할 때 도구 노드로 이동하거나, 도구가 필요 없으면 바로 종료하게 됨.

# 도구 호출 후 챗봇으로 돌아가는 경로 설정.
graph_builder.add_edge("tools", "chatbot")

# 그래프의 시작점 설정.
graph_builder.add_edge(START, "chatbot")

# 그래프 완성 및 컴파일
#graph = graph_builder.compile()

# 시각화
#from IPython.display import Image, display

#try:
#    display(Image(graph.get_graph().draw_mermaid_png()))
#except Exception:
#    pass



In [15]:
# 챗봇에게 질문하기
# 사용자가 입력한 질의문을 그래프에 전달하고, 그래프가 실행하는 동안 생성되는 응답을 스트리밍 방식으로 출력함.
# 응답 생성 과정에서 도구 사용이 필요하면 랭그래프가 정의된 흐름에 따라 외부 도구를 자동으로 호출한 뒤 최종 응답을 반환함.
##########################################################################################################
#def stream_graph_updates(user_input: str):
#    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
#        for value in event.values():
#            print("Asistant:", value["messages"][-1].content) 
##########################################################################################################            

# 이 함수는 graph.stream() 을 사용해 랭그래프의 워크플로우를 단계별로 실행하고, 실행 과정에서 생성되는 중간 결과를 이벤트 형태로 받아옴.
# 각 이벤트에는 하나 이상의 노드 실행 결과가 포함돼 있으며 그 안에서 챗봇이 생성한 메시지를 확인할 수 있음.
# 메시지는 리스트 형태로 관리되며, 이 중에서 가장 마지막 항목이 현재 단계에서 생성된 최신 응답에 해당함.
# 
# 이러한 구조를 통해 챗봇의 응답을 실시간으로 확인할 수 있음. 다음은 이 함수를 사용해 대화를 주고 받는 예시임.
##########################################################################################################
#try:
#    while True:
#        user_input = input("User: ").strip()

#        # 빈입력 처리
#        if not user_input:
#            continue

#        # 종료조건
#        if user_input.lower() in ["quit", "exit", "q"]:
#            print("Goodbye")
#            break
#        stream_graph_updates(user_input)
#except (KeyboardInterrupt, EOFError):
#    # 노트북에서 Interrupt(정지버튼) 누를 때 안전하게 종료
#    print("\n 대화가 중단되었습니다.")
##########################################################################################################    

In [ ]:
## 메모리 기능
# 앞서 챗봇은 검색 도구를 활용해 실시간 정보를 찾아주는 기능을 갖췄지만 여전히 한가지 중요한 기능이 빠져 있음.
# 바로 대화의 맥락을 기억하는 기능, 즉 멀티턴(Multi-Turn)대화 임.

# 이러한 문제를 해결하기 위해 랭그래프는 체크포인팅(checkpointing)기능을 제공함.

# 랭그래프에서는 상태를 자동으로 저장하고 복원할 수 있도록 체크포인터(checkpointer)를 제공함.
# 체크포인터를 사용하면 각 노드는 실행될때마다 상태가 저장되며, 이후 동일한 thread_id 로 그래프를 다시 실행하면 이전 상태를 불러와 대화를 이어갈수 있음.

# 1) 랭그래프의 체크 포인트
# 01. 메모리 체크포인트 생성.
# 간단하게 메모리를 기반으로 상태를 저장하는 MemoryServer 를 사용함.
# 실제 운영환경에서는 SQLite나 PostgreSQL을 사용하는 체크포인트로 교체하면 됨.

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# 02. 그래프 컴파일 시 메모리 연동.
# compile() 로 기존에 구성한 그래프를 컴파일할때 checkpointer인자를 통해 앞서 생성한 memory를 함께 전달하면 됨.
# 이렇게 하면 langgraph는 각 노드 실행후 상태를 자동으로 저장하며, 나중에 같은 thread_id 로 호출하면 이전 대화 상태를 이어받아 응답을 할수 있음.
graph = graph_builder.compile(checkpointer = memory)

# 03. 챗봇과 대화 시작.
# 이제 챗봇과 대화를 시작할 수있음.
# 대화는 thread_id 값을 기준으로 상태가 구분되며, 동일한 thread_id로 여러차례 호출하면 이전 상태를 기억하고 대화를 이어감.
config = {"configurable": {"thread_id":1}}
user_input = "안녕하세요! 제이름은 랭체인입니다."

# 랭그래프에서 그래프를 실행함. 그 과정에서 발생하는 실시간 데이터 흐름(이벤트나 상태 변화)을 스트리밍 형태로 받아오는 메서드
events = graph.stream(
    {"messages": [{"role":"user", "content": user_input}]},
    config,
    stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print() # pretty_print() : 파이썬 객체(딕셔너리, 리스트 등)을 사람이 읽기 좋게 들여쓰기와 줄바꿈을 통해 예쁘게 출력하는 기능.

# 04. 후속 질문하기
# 위 예시에서 챗봇은 사용자의 이름을 기억했음. 이어서 후속 질문을 해보면 챗봇이 이전에 들은 내용을 기억하고 있는지 확인할 수 있음.
# 같은 thread_id 를 사용했기 때문에 챗봇은 이전 대화 내용을 기억하고 '랭체인'이라는 이름을 언급할 수 있음.
user_input = "내 이름은 뭐였나요?"

events = graph.stream(
    {"messages": [{"role":"user", "content": user_input}]},
    config,
    stream_mode = "values"
)

for event in events:
    event["messages"][-1].pretty_print()

# 05. 현재 상태 확인하기
# 저장된 상태를 직접 확인하고 싶다면 graph.get_state() 메서드를 사용하면 됨.
snapshot = graph.get_state(config)
print(snapshot)

In [ ]:
# 이 스냅샷 객체에는 현재까지의 메시지 내역, 다음에 실행될 노드 정보, 그리고 내부 메타데이터가 함께 포함돼 있음.
# 그래프 실행후 어떤 노드가 다음에 수행될 예정인지 확인하고 싶다면 다음과 같이 snapshot.next 값을 출력하면 됨.
snapshot.next # 다음 실행할 노드(대화가 끝났다면 빈 튜플)

# 만약 현재까지의 메시지 리스트가 궁금하다면 다음 코드를 통해 확인할 수 있음.
snapshot.values # 현재까지의 메시지 리스트

# 이렇게 checkpointer 기능을 통해 대화 상태를 저장하고 이어갈 수 있는 챗봇을 만들었음.
# 단순히 일시적 메모리 대신 상태 기반 대화 흐름을 관리할 수 있기 때문임.
# 에러 복구, 사람 개입, 시간차 인터렉션 등 보다 복잡한 시나리오도 다룰 수 있음.